# 04 — V-Order Demo (visible proof)

## Story beat: "Pay a little on write, save on every read"

Contoso's `fact_sales` will be scanned millions of times by Power BI and the warehouse. **V-Order** rearranges Parquet data at write time so engines read less data per query. This notebook runs a **controlled experiment**: same data, two tables — with and without V-Order — then compares **file size** and **query time**.

---

## How V-Order works (conceptual)

1. **Sort** rows on columns BI filters often (store, date).
2. **Dictionary-encode** repeated values (region names, categories).
3. **Row-group layout** lets VertiScan / SQL skip whole groups that don't match the filter.

Cost: slightly slower writes + `OPTIMIZE` job. Benefit: smaller files, faster targeted aggregations.

> **Presenter note:** On this tiny demo dataset the gap may be modest. Say *"At billion-row scale and with Direct Lake, V-Order is where you win — we prove the mechanism here."*

In [ ]:
%run 00_config

## Step 1 — Prepare source data

We copy `silver.fact_sales` into a `demo` schema for side-by-side comparison. Production would never duplicate — this is teaching only.

In [ ]:
import time

spark.sql("CREATE SCHEMA IF NOT EXISTS demo")
src = spark.table(f"{SILVER_SCHEMA}.fact_sales")

## Step 2 — Write two identical tables (V-Order on vs off)

| Table | V-Order | Notes |
| --- | --- | --- |
| `demo.fact_no_vorder` | **Off** | Default Parquet layout |
| `demo.fact_vorder` | **On** + `OPTIMIZE` | Rewritten in V-Order layout |

In [ ]:
spark.conf.set("spark.sql.parquet.vorder.default", "false")
src.write.mode("overwrite").format("delta").saveAsTable("demo.fact_no_vorder")

spark.conf.set("spark.sql.parquet.vorder.default", "true")
src.write.mode("overwrite").format("delta").saveAsTable("demo.fact_vorder")
spark.sql("OPTIMIZE demo.fact_vorder")
print("both tables written")

## Step 3 — Compare on-disk size

`DESCRIBE DETAIL` returns Delta metadata including total size and file count. V-Order often reduces bytes **and** file count after OPTIMIZE.

In [ ]:
def detail(name):
    d = spark.sql(f"DESCRIBE DETAIL {name}").collect()[0]
    return d["sizeInBytes"], d["numFiles"]

for t in ["demo.fact_no_vorder", "demo.fact_vorder"]:
    size, files = detail(t)
    print(f"{t:24s} size={size/1024/1024:8.2f} MB  files={files}")

## Step 4 — Compare aggregation query time

Same query on both tables: `SUM(net_amount) GROUP BY store_id`. We warm the cache first, then time one run.

> **Warehouse note:** In `wh_retail` (Module 2), V-Order is **on by default** — you don't opt in manually.

In [ ]:
def timeit(name):
    q = f"SELECT store_id, SUM(net_amount) FROM {name} GROUP BY store_id"
    spark.sql(q).collect()  # warm
    t0 = time.time()
    spark.sql(q).collect()
    return round(time.time() - t0, 3)

print("no V-Order :", timeit("demo.fact_no_vorder"), "s")
print("V-Order    :", timeit("demo.fact_vorder"), "s")
print("\nNote: gap grows with data volume and column selectivity.")

---

## ✅ Success looks like

- V-Order table is same size or smaller; query time same or faster.
- Audience understands *why* we OPTIMIZE gold tables before Direct Lake.

**Next:** `05_shortcuts` — federate external data without copying.